## Step 2: Convert ONNX → TFLite

Convert the ONNX model to TensorFlow Lite format for OpenMV deployment.

In [5]:
%pip install ai-edge-litert
%pip install sng4onnx
import os
import shutil
import onnx2tf

# --------------------------------------------------
# 1️⃣ Paths
# --------------------------------------------------

ONNX_PATH = "models/volume_predictor.onnx"
OUTPUT_DIR = "models/tf_model"
FINAL_TFLITE_PATH = "models/volume_predictor.tflite"

# Make sure ONNX exists
if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(f"ONNX file not found at: {ONNX_PATH}")

print(f"✓ Found ONNX file: {ONNX_PATH}")
print(f"  Size: {os.path.getsize(ONNX_PATH)/(1024*1024):.2f} MB")

# --------------------------------------------------
# 2️⃣ Convert ONNX → TensorFlow / TFLite
# --------------------------------------------------

print("\n🔄 Converting ONNX → TensorFlow → TFLite...")

onnx2tf.convert(
    input_onnx_file_path=ONNX_PATH,
    output_folder_path=OUTPUT_DIR,
    non_verbose=False,
)

print("✓ Conversion complete")

# --------------------------------------------------
# 3️⃣ Locate Generated TFLite File
# --------------------------------------------------

tflite_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".tflite")]

if not tflite_files:
    raise RuntimeError(
        f"No .tflite file found in {OUTPUT_DIR}\n"
        f"Files found: {os.listdir(OUTPUT_DIR)}"
    )

# ⚠️ IMPORTANT: Use the Float32 version as the baseline!
# onnx2tf generates both float32 and float16 versions
tflite_float32 = os.path.join(OUTPUT_DIR, "volume_predictor_float32.tflite")

if os.path.exists(tflite_float32):
    shutil.copy2(tflite_float32, FINAL_TFLITE_PATH)
    print(f"\n✓ Using Float32 baseline: {tflite_float32}")
else:
    # Fallback to first file if float32 not found
    tflite_src = os.path.join(OUTPUT_DIR, tflite_files[0])
    shutil.copy2(tflite_src, FINAL_TFLITE_PATH)
    print(f"\n⚠️  Float32 version not found, using: {tflite_files[0]}")

print(f"✓ Final TFLite model saved to: {FINAL_TFLITE_PATH}")
print(f"  File size: {os.path.getsize(FINAL_TFLITE_PATH)/1024:.2f} KB")

# Show what onnx2tf generated
print(f"\n📦 onnx2tf generated files:")
for f in sorted(tflite_files):
    fpath = os.path.join(OUTPUT_DIR, f)
    print(f"  • {f}: {os.path.getsize(fpath)/1024:.2f} KB")

print("\n🎉 Conversion pipeline complete.")


You should consider upgrading via the 'd:\Code\lab\volume_convert_env\Scripts\python.exe -m pip install --upgrade pip' command.
You should consider upgrading via the 'd:\Code\lab\volume_convert_env\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.

✓ Found ONNX file: models/volume_predictor.onnx
  Size: 6.34 MB

🔄 Converting ONNX → TensorFlow → TFLite...

Model optimizing started ============================================================
Traceback (most recent call last):
  File "d:\Code\lab\volume_convert_env\lib\site-packages\onnx2tf\onnx2tf.py", line 716, in convert
    result = subprocess.check_output(
  File "C:\Users\VHC.LAPTOP-S7TBJVGH\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 420, in check_output
    return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
  File "C:\Users\VHC.LAPTOP-S7TBJVGH\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 524, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['onnxsim', 'models/volume_predictor.onnx', 'models/volume_predictor.onnx']' returned non-zero exit status 1.


Automatic generation of each OP name started ===========

## Step 3: Quantize TFLite Model (Reduce Size)

Apply quantization to reduce model size with minimal accuracy loss:
- **Float16**: ~50% smaller, virtually no accuracy loss
- **Dynamic Range Int8**: ~75% smaller, slight accuracy loss (easiest int8 option)
- **Full Int8**: ~75% smaller, requires calibration data
- **Int8 + Pruning**: Can get 80-90% smaller with additional techniques

**Additional size reduction options:**
1. **Model Pruning** - Remove low-importance weights before conversion
2. **Smaller Architecture** - Retrain with fewer layers/channels
3. **Knowledge Distillation** - Train a tiny model to mimic this one

In [7]:
import tensorflow as tf
import numpy as np
import os

# Define paths (in case this cell is run independently)
OUTPUT_DIR = "models/tf_model"
FINAL_TFLITE_PATH = "models/volume_predictor.tflite"

# Load the SavedModel directory generated by onnx2tf
SAVED_MODEL_DIR = OUTPUT_DIR

# --------------------------------------------------
# 1️⃣ Float16 Quantization (~50% size reduction)
# --------------------------------------------------

print("\n🔄 Applying Float16 quantization...")

converter_fp16 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]

tflite_fp16_model = converter_fp16.convert()

# Save Float16 model
fp16_path = "models/volume_predictor_float16.tflite"
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16_model)

print(f"✓ Float16 model saved: {fp16_path}")
print(f"  Size: {os.path.getsize(fp16_path)/1024:.2f} KB")
print(f"  Reduction: {(1 - os.path.getsize(fp16_path)/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}%")

# --------------------------------------------------
# 2️⃣ Dynamic Range Int8 Quantization (~75% size reduction, EASIEST)
# --------------------------------------------------

print("\n🔄 Applying Dynamic Range Int8 quantization (no calibration needed)...")

converter_dynamic = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_dynamic.optimizations = [tf.lite.Optimize.DEFAULT]
# No representative dataset needed - weights are quantized, activations stay float

try:
    tflite_dynamic_model = converter_dynamic.convert()
    
    dynamic_path = "models/volume_predictor_dynamic_int8.tflite"
    with open(dynamic_path, 'wb') as f:
        f.write(tflite_dynamic_model)
    
    print(f"✓ Dynamic Int8 model saved: {dynamic_path}")
    print(f"  Size: {os.path.getsize(dynamic_path)/1024:.2f} KB")
    print(f"  Reduction: {(1 - os.path.getsize(dynamic_path)/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}%")
except Exception as e:
    print(f"⚠️  Dynamic quantization failed: {e}")

# --------------------------------------------------
# 3️⃣ Full Int8 Quantization (~75% size, needs representative data)
# --------------------------------------------------

print("\n🔄 Applying Full Int8 quantization (with calibration)...")

# Create representative dataset from training data
# This helps the converter determine optimal quantization parameters
def representative_dataset():
    """Generate sample inputs for quantization calibration"""
    # Using random data - replace with actual training images for best results
    for _ in range(100):
        # Shape: (1, 1, 320, 160) - batch, channels, height, width
        data = np.random.rand(1, 1, 320, 160).astype(np.float32)
        yield [data]

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset

# For full int8 quantization (inputs and outputs too)
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8  # or tf.int8
converter_int8.inference_output_type = tf.uint8  # or tf.int8

try:
    tflite_int8_model = converter_int8.convert()
    
    # Save Int8 model
    int8_path = "models/volume_predictor_int8.tflite"
    with open(int8_path, 'wb') as f:
        f.write(tflite_int8_model)
    
    print(f"✓ Full Int8 model saved: {int8_path}")
    print(f"  Size: {os.path.getsize(int8_path)/1024:.2f} KB")
    print(f"  Reduction: {(1 - os.path.getsize(int8_path)/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}%")
except Exception as e:
    print(f"⚠️  Full Int8 quantization failed: {e}")
    print("   This is normal if the model has operations not supported in int8")

# --------------------------------------------------
# 4️⃣ Summary
# --------------------------------------------------

print("\n" + "="*60)
print("📊 Model Size Comparison")
print("="*60)
print(f"Original (Float32):      {os.path.getsize(FINAL_TFLITE_PATH)/1024:>8.2f} KB  (baseline)")
print(f"Float16 Quantized:       {os.path.getsize(fp16_path)/1024:>8.2f} KB  ({(1-os.path.getsize(fp16_path)/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}% smaller)")
if os.path.exists("models/volume_predictor_dynamic_int8.tflite"):
    print(f"Dynamic Int8:            {os.path.getsize('models/volume_predictor_dynamic_int8.tflite')/1024:>8.2f} KB  ({(1-os.path.getsize('models/volume_predictor_dynamic_int8.tflite')/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}% smaller) ⭐ RECOMMENDED")
if os.path.exists("models/volume_predictor_int8.tflite"):
    print(f"Full Int8:               {os.path.getsize('models/volume_predictor_int8.tflite')/1024:>8.2f} KB  ({(1-os.path.getsize('models/volume_predictor_int8.tflite')/os.path.getsize(FINAL_TFLITE_PATH))*100:.1f}% smaller)")
print("="*60)
print("\n💡 Recommended: Dynamic Int8 for best balance of size, accuracy, and ease of use")
print("💡 For even smaller: See optional pruning techniques below")



🔄 Applying Float16 quantization...
✓ Float16 model saved: models/volume_predictor_float16.tflite
  Size: 3252.36 KB
  Reduction: 49.9%

🔄 Applying Dynamic Range Int8 quantization (no calibration needed)...
✓ Dynamic Int8 model saved: models/volume_predictor_dynamic_int8.tflite
  Size: 1630.95 KB
  Reduction: 74.9%

🔄 Applying Full Int8 quantization (with calibration)...
⚠️  Full Int8 quantization failed: Attempting to resize dimension 1 of tensor 0 with value 320 to 1. ResizeInputTensorStrict only allows mutating unknown dimensions identified by -1.
   This is normal if the model has operations not supported in int8

📊 Model Size Comparison
Original (Float32):       6496.70 KB  (baseline)
Float16 Quantized:        3252.36 KB  (49.9% smaller)
Dynamic Int8:             1630.95 KB  (74.9% smaller) ⭐ RECOMMENDED

💡 Recommended: Dynamic Int8 for best balance of size, accuracy, and ease of use
💡 For even smaller: See optional pruning techniques below


## Optional: Int8 Quantization with Real Training Data

For better Int8 quantization accuracy, use actual training images instead of random data. 
This ensures the quantization parameters are optimized for your actual data distribution.

In [ ]:
# Optional: Better Int8 quantization using real training data
# Uncomment and run if you want the best Int8 model

'''
import torch
from PIL import Image

# Load some training images (adjust paths as needed)
def representative_dataset_real():
    """Use real training images for quantization calibration"""
    import glob
    
    # Get list of training images
    image_paths = []
    for attempt in range(1, 11):  # Attempt1 to Attempt10
        pattern = f"VolPicturesRound1/Attempt{attempt}/*.jpeg"
        image_paths.extend(glob.glob(pattern)[:10])  # Use 10 images per attempt
    
    print(f"Using {len(image_paths)} real images for quantization calibration...")
    
    for img_path in image_paths[:100]:  # Use up to 100 images
        try:
            # Load and preprocess image
            img = Image.open(img_path).convert('L')  # Grayscale
            img = img.resize((160, 320))  # Match model input
            img_array = np.array(img, dtype=np.float32) / 255.0
            img_array = np.expand_dims(img_array, axis=(0, 1))  # Add batch and channel dims
            yield [img_array]
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            continue

# Convert with real data
converter_int8_real = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_int8_real.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8_real.representative_dataset = representative_dataset_real
converter_int8_real.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8_real.inference_input_type = tf.uint8
converter_int8_real.inference_output_type = tf.uint8

tflite_int8_real = converter_int8_real.convert()

# Save
int8_real_path = "models/volume_predictor_int8_calibrated.tflite"
with open(int8_real_path, 'wb') as f:
    f.write(tflite_int8_real)

print(f"✓ Calibrated Int8 model saved: {int8_real_path}")
print(f"  Size: {os.path.getsize(int8_real_path)/1024:.2f} KB")
'''

print("ℹ️  Uncomment the code above to create an Int8 model with real training data")